# Load Documents

In [1]:
import os
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_core.documents import Document
import re
import hashlib
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerCorrectness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

/Users/adrieljie2708/Desktop/RAG_System/.venv-arm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DOCS_PATH="./docs/"
DB_PATH = "./db/ChromaDB/"
COLLECTION_NAME = "books_dedup"

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name = "BAAI/bge-base-en-v1.5")
db = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=DB_PATH,
    collection_metadata={"hnsw:space": "cosine"}
)

W1019 23:04:21.225086 30982 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15833.57it/s]


In [4]:
def preprocess_text(text):
    match = re.search(
        r"^\*\*\* START OF THE PROJECT GUTENBERG EBOOK[^\n]*\n"
        r"(.*?)"
        r"^\*\*\* END OF THE PROJECT GUTENBERG EBOOK[^\n]*",
        text,
        flags=re.MULTILINE | re.DOTALL
    )
    if match:
        text = match.group(1)
    text = re.sub(r"(?m)^[ \t]*\*(?:[ \t]*\*)*[ \t]*$", "", text)
    text = re.sub(r"(?m)^[ \t]*\[Illustration\][ \t]*$", "", text)
    text = re.sub(r"\n(?:[ \t]*\n){2,}", "\n\n", text)
    return text.strip()

In [5]:
def load_document(folder_path, db):
    print(f'Loading documents...')

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f'\nFolder {folder_path} does not exist. Create folder and add your favorite books.')

    documents =[]
    hashed = set()
    for filename in sorted(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)

        if not os.path.isfile(file_path):
             print(f'\nNo documents found in {folder_path}.')
             continue
        if not filename.endswith('.txt'):
             print(f'\n{filename} document type is not supported.')
             continue

        with open(file_path, 'rb') as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()

        stored = db.get(where={'hash_file': file_hash}, include=[])

        if stored['ids']:
            print(f'\n{filename} has been stored in database.')
            continue

        if file_hash in hashed:
            print(f"\n{filename}'s content is detected for duplication.")
            continue
        loader = TextLoader(file_path=file_path, encoding="utf-8")
        documents_load = loader.load()

        for document in documents_load:
                text = document.page_content
                document.metadata['hash_file'] = file_hash
                for key, label in [('title', 'Title'), ('author', 'Author'), ('release_date', 'Release date'), ('language', 'Language')]:
                    match = re.search(rf"(?im)^{label}:[ \t]*(.+)$", text)
                    document.metadata[key] = match.group(1).strip() if match else "Unknown"

                document.metadata['release_date'] = document.metadata['release_date'].split("[")[0].strip()
                document.page_content = preprocess_text(text)

        hashed.add(file_hash)
        documents.extend(documents_load)
    if not documents:
        print("\nNo new documents to process.")
        return []

    for i, doc in enumerate(documents[:2]):
        print(f"\nDocument {i+1}: ")
        print(f"Source: {doc.metadata['source']}")
        print(f'Content length: {len(doc.page_content)}')
        print(f'Content preview: {doc.page_content[:100]}')
        print(f'Metadata: {doc.metadata}')

    return documents

In [6]:
def split_documents(documents):
    print("Splitting documents into chunks.")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=0,
        separators=['\n\n','\n',' ','']
    )

    chunks = text_splitter.split_documents(documents)

    positions = {}
    for chunk in chunks:
        source = chunk.metadata['source']
        position = positions.get(source, 0)
        chunk.metadata['chunk_id'] = position
        positions[source] = position+1

    if chunks:

        for i, chunk in enumerate(chunks):
            print(f"\nChunk ID: {chunk.metadata['chunk_id']}")
            print(f"Source: {chunk.metadata['source']}")
            print(f'Length: {len(chunk.page_content)}')
            print(f'Content:')
            print(chunk.page_content)
            print("\n")

        if len(chunks) > 5:
            print(f'\n and {len(chunks) - 5} more chunks')
    return chunks

In [7]:
def menu1():
    documents = load_document(DOCS_PATH, db)

    if documents:
        chunks = split_documents(documents)
        db.add_documents(documents=chunks)
        print(f'{len(documents)} new documents has been stored to database.')

In [8]:
documents = load_document(DOCS_PATH, db)
chunks = split_documents(documents)

Loading documents...

alice_in_wonderland.txt has been stored in database.

frankenstein.txt has been stored in database.

pride_and_prejudice.txt has been stored in database.

sherlock_holmes.txt has been stored in database.

No new documents to process.
Splitting documents into chunks.


# Retrieval

In [9]:
pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    device = 0,
    pipeline_kwargs={
        "max_new_tokens": 512,
        "do_sample": False,
        "return_full_text": False
    }
)

model = ChatHuggingFace(llm=pipeline)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8833.14it/s]


In [10]:
def ask_question(question):
    print(f'Question: {question}')
    retriever = db.as_retriever(search_kwargs={"k": 5})
    relevant_docs = retriever.invoke(question)

    print(f'\nFound {len(relevant_docs)} documents')

    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    combined_input = f"""Based on the following documents, please answer this question: {question}
    Documents:
    {context}
    Please provide a clear, helpful answer using only the information provided in the documents. If you don't know the answer, just say that you don't know.
    """

    messages = [SystemMessage(content="You are a question-answering assistant for books. Answer the question using only the provided context. Rules: 1. Read all relevant passages before answering. 2. Identify all important reasons related to the question. 3. Combine information from different passages when necessary. 4. Consider the order of events in the story. 5. Keep the answer concise but complete. 6. Do not invent information.")] + [HumanMessage(content=combined_input)]


    if not relevant_docs:
      print("I couldn't find relevant information.")
      return answer, []
    
    response = model.invoke(messages)
    answer = response.content

    print(f'Answer: {answer}')

    for doc in relevant_docs:
      print(f'\nSource: {doc.metadata["title"]} by {doc.metadata["author"]} –– Chunk {doc.metadata["chunk_id"]}')
      # preview = doc.page_content[:250].replace("\n", " ")
      # if len(doc.page_content) > 250:
      #   preview += "..."
      print(f'Evidence: {doc.page_content}')

    return answer, relevant_docs

# Retrieval Evaluation

In [11]:
eval_dataset = [
    {
        "question": "Why does Alice follow the White Rabbit?",
        "ref_chunks": {("alice_in_wonderland.txt", 2)}
    },
    {
        "question": "What opens the small door behind the curtain?",
        "ref_chunks": {("alice_in_wonderland.txt", 8)}
    },
    {
        "question": "Why can't Alice climb the table leg to reach the key?",
        "ref_chunks": {("alice_in_wonderland.txt", 13)}
    },
    {
        "question": "Who wakes Alice at the end of her adventure?",
        "ref_chunks": {("alice_in_wonderland.txt", 167)}
    },
    {
        "question": (
            "Who does Sherlock Holmes introduce as his friend "
            "and associate to the veiled lady?"
        ),
        "ref_chunks": {("sherlock_holmes.txt", 425)}
    }
]

In [22]:
def retrieval_evaluation(db, eval_dataset):
    retriever = db.as_retriever(search_kwargs={"k": 5})
    results = []

    for data in eval_dataset:
        question = data['question']
        ref_chunk = data['ref_chunks']
        relevant_docs = retriever.invoke(question)
        first_relevant = None

        print(f'Question: {question}')
        for rank, doc in enumerate(relevant_docs, start=1):
            filename = os.path.basename(doc.metadata['source'])
            chunk_id = int(doc.metadata['chunk_id'])

            ref = (filename, chunk_id) in ref_chunk

            if ref and first_relevant is None:
                first_relevant = rank
            found = "(Referenced)" if ref else ""
            print(f"{rank}. {filename} — Chunk {chunk_id} {found}")

        if first_relevant is not None:
            hit = 1
            rr = 1/first_relevant
        else:
            hit = 0
            rr = 0

        results.append({
            'question': question,
            'first_relevant': first_relevant,
            'hit': hit,
            'rr': rr
        })

        print(f"Hit: {hit}")
        print(f"Reciprocal rank: {rr:.3f}")

    hit_rate = sum(result['hit'] for result in results) / len(results)
    mrr = sum(result['rr'] for result in results) / len(results)

    print("\nRetrieval evaluation")
    print(f"Hit Rate: {hit_rate:.2%}")
    print(f"MRR: {mrr:.3f}")

    return results

eval_result = retrieval_evaluation(db, eval_dataset)

Question: Why does Alice follow the White Rabbit?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 41 
3. alice_in_wonderland.txt — Chunk 2 (Referenced)
4. alice_in_wonderland.txt — Chunk 158 
5. alice_in_wonderland.txt — Chunk 141 
Hit: 1
Reciprocal rank: 0.333
Question: What opens the small door behind the curtain?
1. alice_in_wonderland.txt — Chunk 8 (Referenced)
2. sherlock_holmes.txt — Chunk 723 
3. sherlock_holmes.txt — Chunk 52 
4. sherlock_holmes.txt — Chunk 645 
5. alice_in_wonderland.txt — Chunk 9 
Hit: 1
Reciprocal rank: 1.000
Question: Why can't Alice climb the table leg to reach the key?
1. alice_in_wonderland.txt — Chunk 13 (Referenced)
2. alice_in_wonderland.txt — Chunk 8 
3. alice_in_wonderland.txt — Chunk 48 
4. alice_in_wonderland.txt — Chunk 4 
5. alice_in_wonderland.txt — Chunk 3 
Hit: 1
Reciprocal rank: 1.000
Question: Who wakes Alice at the end of her adventure?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 51 
3.

# Generation Evaluation

In [14]:
generation_data = [
    {
        "question": "Why does Alice follow the White Rabbit?",
        "reference_answers": [
            "Alice follows the White Rabbit because she is curious "
            "after seeing it take a watch out of its waistcoat pocket."
        ]
    },
    {
        "question": "What opens the small door behind the curtain?",
        "reference_answers": [
            "A little golden key.",
            "The small door is opened with a little golden key."
        ]
    },
    {
        "question": "Why can't Alice climb the table leg to reach the key?",
        "reference_answers": [
            "Because the table leg is too slippery.",
            "Alice cannot climb the table leg because it is too slippery."
        ]
    },
    {
        "question": "Who wakes Alice at the end of her adventure?",
        "reference_answers": [
            "Her sister.",
            "Alice's sister wakes her at the end of her adventure."
        ]
    },
    {
        "question": (
            "Who does Sherlock Holmes introduce as his friend "
            "and associate to the veiled lady?"
        ),
        "reference_answers": [
            "Dr. Watson.",
            "Sherlock Holmes introduces Dr. Watson "
            "as his friend and associate."
        ]
    }
]

In [ ]:
evaluation_rows = []
for item in generation_data:
    question = item["question"]

    answer, relevant_docs = ask_question(question)

    evaluation_rows.append({
        "user_input": question,
        "response": answer,
        "retrieved_contexts": [doc.page_content for doc in relevant_docs],
        "reference": item["reference_answers"][0]
    })

evaluation_dataset = EvaluationDataset.from_list(evaluation_rows)

Question: Why does Alice follow the White Rabbit?

Found 5 documents
Answer: Alice follows the White Rabbit because she is curious about him and wants to find out more about his whereabouts.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door,

In [18]:
import asyncio
from ragas.llms.base import LangchainLLMWrapper


class LocalRagasWrapper(LangchainLLMWrapper):
    async def agenerate_text(
        self,
        prompt,
        n=1,
        temperature=0.01,
        stop=None,
        callbacks=None
    ):
        return await asyncio.to_thread(
            self.generate_text,
            prompt=prompt,
            n=n,
            temperature=temperature,
            stop=stop,
            callbacks=callbacks
        )

In [19]:
judge_model = model
evaluator_llm = LocalRagasWrapper(judge_model)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

evaluation_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        Faithfulness(),
        AnswerCorrectness()
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(
        max_workers=1,
        max_retries=0
    ),
    raise_exceptions=True
)

scores = evaluation_result.to_pandas()

display(scores[
    ["user_input", "response", "reference",
     "faithfulness", "answer_correctness"]
])

metric_columns = ["faithfulness", "answer_correctness"]

if scores[metric_columns].isna().any().any():
    print("Ada skor yang gagal dihitung. Periksa sebelum menyimpulkan hasil.")
else:
    print("\nAverage scores:")
    print(scores[metric_columns].mean())

/var/folders/3x/mhkm5bcn4_53q1b506vlln3r0000gn/T/ipykernel_30982/1960204343.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)
Evaluating: 100%|██████████| 10/10 [02:16<00:00, 13.64s/it]


,user_input,response,reference,faithfulness,answer_correctness
0,Why does Alice follow the White Rabbit?,Alice follows the White Rabbit because she is ...,Alice follows the White Rabbit because she is ...,1.000000,0.717572
1,What opens the small door behind the curtain?,Alice opens the small door behind the curtain.,A little golden key.,1.000000,0.899673
2,Why can't Alice climb the table leg to reach t...,Alice cannot climb the table leg to reach the ...,Because the table leg is too slippery.,0.000000,0.656221
3,Who wakes Alice at the end of her adventure?,Alice doesn't wake anyone at the end of her ad...,Her sister.,0.333333,0.090247
4,Who does Sherlock Holmes introduce as his frie...,Sherlock Holmes introduces Dr. Watson as his f...,Dr. Watson.,1.000000,0.914123



Average scores:
faithfulness          0.666667
answer_correctness    0.655567
dtype: float64


# Main

In [ ]:
def main():
  print('Library AI Chatbot!')
  while True:
    print("\nMenu:\n1. Load Data\n2. Ask LLM\n3. Exit\n")
    choice = int(input("Enter your choice: "))

    if choice==1:
      menu1()

    elif choice == 2:
      while True:
        question = input('\nYour question: ')

        if question.lower()=='quit' or question=='Quit':
          print('\nGoodbye')
          break

        ask_question(question)
        print('\nType Quit or quit to back to main menu.')
    elif choice == 3:
      print('Thank you for using Library AI Chatbot!')
      break

In [ ]:
if __name__ == "__main__":
  main()

Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

Loading documents...

Document 1: 
Source: ./docs/alice_in_wonderland.txt
Content length: 144117
Content preview: Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 C
Metadata: {'source': './docs/alice_in_wonderland.txt', 'hash_file': '01b38ea4c710a84bc18d0bd41271a5a1a92b94e97b2812f4dece97d4a694725e', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'release_date': 'June 27, 2008', 'language': 'English'}

Document 2: 
Source: ./docs/frankenstein.txt
Content length: 419194
Content preview: Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley

 CONTENTS

 Lette
Metadata: {'source': './docs/frankenstein.txt', 'hash_file': '7810cd483cffcf2cc8a1d8f0d5807931e69d4f48cd14149b8c76f88af82fead3', 'title': 'Frankenstein; or, the modern prometheus', 'author': 'Mary Wollstonecraft Shelley', 'release_date': 'October 1, 1993', 'language': 'Engl